In [ ]:
import pandas as pd
import pandas_ta as ta
import matplotlib.pyplot as plt
import pymannkendall as mk
import numpy as np

def rolling_modified_zscore(data: pd.Series, window: int) -> pd.Series:
    """
    Compute the rolling modified Z‐score of a pandas Series.
    
    Parameters
    ----------
    data : pd.Series
        Input time series.
    window : int
        Rolling window size (number of points) to compute median and MAD.
    
    Returns
    -------
    pd.Series
        Series of modified Z‐scores, aligned with `data`.  The first (window−1)
        values will be NaN.
    """
    # rolling median
    roll_med = data.rolling(window=window, min_periods=window).median()
    
    # rolling MAD: median(|x - roll_med|)
    def mad(arr: np.ndarray) -> float:
        med = np.median(arr)
        return np.median(np.abs(arr - med))
    
    roll_mad = (
        data
        .rolling(window=window, min_periods=window)
        .apply(mad, raw=True)
    )
    
    # modified Z‐score
    mz = 0.6745 * (data - roll_med) / roll_mad
    
    return mz

def calculate_technical_indicators(df: pd.DataFrame, window: int) -> pd.DataFrame:
    """
    Calculate a suite of technical indicators on OHLC price data.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns ['Open','High','Low','Close'] and DateTime index.
    window : int
        Rolling window length for indicators and scores.

    Returns
    -------
    pd.DataFrame
        Original DataFrame with added columns:
        - close_ret, ret_z, ret_mod_z, pos_close_frac
        - high_ret, pos_high_frac
        - low_ret, pos_low_frac
        - chop (Choppiness Index)
        - DMP_{window}, DMN_{window}, ADX_{window}
        - atr_{window}
        - close_z
        - ema_7, ema_14, ema_21, ema_28, ema_score
        - BBM_{window}, BBHI_{window}, BBLO_{window}, BBW_{window}, BBP_{window}
        - rsi_{window}
    """

    # 1. Close return
    df['close_ret'] = df['Close'].pct_change()

    # 2. Z-score of returns (rolling)
    df['ret_z'] = ta.zscore(df['close_ret'], length=window)

    # 3. Modified Z-score of returns (rolling)
    df['ret_mod_z'] = rolling_modified_zscore(df['close_ret'], window)

    # 4. Positive close-return fraction
    df['pos_close_frac'] = (df['close_ret'].ge(0).rolling(window).sum() / window) * 100

    # 5. High return & positive high-return fraction
    df['high_ret'] = df['High'].pct_change()
    df['pos_high_frac'] = (df['high_ret'].ge(0).rolling(window).sum() / window) * 100

    # 6. Low return & positive low-return fraction
    df['low_ret'] = df['Low'].pct_change()
    df['pos_low_frac'] = (df['low_ret'].ge(0).rolling(window).sum() / window) * 100

    # 7. Choppiness Index
    df['chop'] = ta.chop(df['High'], df['Low'], df['Close'], length=window)

    # 8. ADX, +DI, -DI
    adx_df = ta.adx(df['High'], df['Low'], df['Close'], length=window)
    df = df.join(adx_df)

    # 9. ATR
    df[f'atr_{window}'] = ta.atr(df['High'], df['Low'], df['Close'], length=window)

    # 10. Z-score of close prices
    df['close_z'] = ta.zscore(df['Close'], length=window)

    # 11. Exponential Moving Averages
    ema_periods = [7, 14, 21, 28]
    for p in ema_periods:
        df[f'ema_{p}'] = ta.ema(df['Close'], length=p)

    # 12. EMA score: pairwise comparisons
    def compute_ema_score(row):
        score = 0
        for i in range(len(ema_periods)):
            for j in range(i+1, len(ema_periods)):
            ema_i = row[f'ema_{ema_periods[i]}']
            ema_j = row[f'ema_{ema_periods[j]}']
            if pd.isna(ema_i) or pd.isna(ema_j):
                continue
            diff_pct = (ema_i - ema_j) / ema_j * 100
            if diff_pct >= 0.25:
                score += 1
            elif diff_pct <= -0.25:
                score -= 1
            # else: do nothing
        return score

    df['ema_score'] = df.apply(compute_ema_score, axis=1)

    # 13. Bollinger Bands
    bb = ta.bbands(df['Close'], length=window)
    df = df.join(bb)

    # 14. RSI
    df[f'rsi_{window}'] = ta.rsi(df['Close'], length=window)

    return df

def rolling_mann_kendall_full(df: pd.DataFrame, column: str, window: int) -> pd.DataFrame:
    """
    Compute rolling Mann–Kendall original_test over a specified window,
    returning trend label, p-value, and Tau.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with your time series.
    column : str
        Name of the column to test (e.g. 'Close').
    window : int
        Rolling window size.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['trend','p','tau'], indexed same as df.
    """
    trends, ps, taus = [], [], []
    series = df[column]

    for i in range(len(series)):
        if i < window - 1:
            # not enough data yet
            trends.append(np.nan)
            ps.append(np.nan)
            taus.append(np.nan)
            continue

        window_data = series.iloc[i - window + 1 : i + 1].dropna().values
        if len(window_data) < window:
            trends.append(np.nan)
            ps.append(np.nan)
            taus.append(np.nan)
        else:
            res = mk.original_test(window_data, alpha=0.05)
            trends.append(res.trend)  # 'increasing','decreasing' or 'no trend'
            ps.append(res.p)
            taus.append(res.Tau)

    return pd.DataFrame({
        'trend': trends,
        'p':       ps,
        'tau':   taus
    }, index=df.index)

        
# Example usage:
# data = pd.read_csv('your_data.csv', parse_dates=['Date'], index_col='Date')
# indicators_df = calculate_technical_indicators(data, window=14)


# read ohlc data from the saved CSV file
df = pd.read_csv('/home/qa/runtime/data/analysis_out/daily_ohlc_with_signals.csv')
# remove timezone information from the 'Date' column
df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)

# filter df for a symbol LUPIN
df = df[df['Symbol'] == 'LUPIN']

# drop open, high, low, close columns 
df['Open'] = df['Adj Open']
df['High'] = df['Adj High']
df['Low'] = df['Adj Low']
df['Close'] = df['Adj Close']
df = df.drop(columns=['Adj Open', 'Adj High', 'Adj Low', 'Adj Close'])

# round OHLC to 2 decimal places
df['Open'] = df['Open'].round(2)
df['High'] = df['High'].round(2)
df['Low'] = df['Low'].round(2)
df['Close'] = df['Close'].round(2)


df = calculate_technical_indicators(df, window=14)
# Calculate rolling Mann-Kendall trend test for 'Close' prices
mk_results = rolling_mann_kendall_full(df, 'Close', window=14)
# Add Mann-Kendall results to the DataFrame
df = df.join(mk_results)
# Save the DataFrame with indicators and Mann-Kendall results
df.to_csv('/home/qa/runtime/data/analysis_out/lupin_technical_indicators.csv', index=False)